# Insurance Approval Modeling with WoE and IV

This notebook presents the final project as a clear case study rather than a scratch workbook. The goal is to build an interpretable model for insurance approval decisions in a setting where positive cases are rare and several predictors are difficult to encode cleanly.


In [ ]:
%pip install optbinning imbalanced-learn joblib

In [ ]:
# 0. Set up
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
from optbinning import OptimalBinning

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")


## 1. Load the data

I start by loading the data from the repository itself. That keeps the notebook portable and avoids tying it to one machine or one local folder layout.

In [ ]:
# 1. Load the training and test data
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

if not (project_root / 'data').exists():
    raise FileNotFoundError("Run this notebook from the repository root or the model/ directory.")

config_dir = project_root / 'configuration'
output_dir = project_root / 'outputs'
output_dir.mkdir(exist_ok=True)

print(f"Using project root: {project_root}")

df_train = pd.read_csv(project_root / 'data' / 'insurance_train.csv')
df_test = pd.read_csv(project_root / 'data' / 'insurance_test.csv')
target_col = 'claim_status'

X_train = df_train.drop(columns=[target_col])
y_train = df_train[target_col]

numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


## 2. Data Exploration

Before fitting anything, I want to confirm the shape of the dataset and the main modeling constraints. In this project, the useful checks are simple:

- how imbalanced the target is;
- which numeric fields are heavily skewed;
- which variables are dominated by zeros;
- which categorical features have high cardinality;
- whether the target shows any basic correlation with the numeric inputs.



### 2a. Dataset snapshot

This table gives the scale of the problem and confirms whether missing-value handling needs to be part of the workflow.

In [ ]:
# Dataset snapshot
dataset_summary = pd.DataFrame(
    {
        'metric': [
            'training_rows',
            'test_rows',
            'input_features',
            'positive_class_rate',
            'missing_values_train',
            'missing_values_test'
        ],
        'value': [
            len(df_train),
            len(df_test),
            X_train.shape[1],
            round(y_train.mean() * 100, 4),
            int(df_train.isna().sum().sum()),
            int(df_test.isna().sum().sum())
        ]
    }
)
dataset_summary


### 2b. Check the target imbalance

The first real risk in this project is class imbalance. If approved claims are rare, accuracy on its own becomes a weak metric and the training strategy needs to pay more attention to minority-class detection.

In [ ]:
# Target imbalance
target_distribution = y_train.value_counts().rename_axis('claim_status').reset_index(name='count')
target_distribution['share'] = target_distribution['count'] / target_distribution['count'].sum()

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=target_distribution, x='claim_status', y='count', ax=ax)
ax.set_title('Training target distribution')
ax.set_xlabel('claim_status')
ax.set_ylabel('count')
for patch, share in zip(ax.patches, target_distribution['share']):
    ax.annotate(f"{share:.2%}", (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()

target_distribution


### 2c. Numeric variables

Several numeric inputs are not well behaved in their raw form. I check skewness and the share of zeros together because both patterns influence how stable the downstream classifier will be.

In [ ]:
# Numeric diagnostics: distribution shape and zero-heavy fields
numeric_profile = df_train[numerical_features].agg(['min', 'median', 'mean', 'max']).T
numeric_profile['skewness'] = df_train[numerical_features].skew()
numeric_profile = numeric_profile.sort_values('skewness', ascending=False)

zero_share = (df_train[numerical_features] == 0).mean().sort_values(ascending=False).rename('zero_share').to_frame()

display(numeric_profile.round(3))
display(zero_share.round(3))


### 2d. Categorical variables

Here I look at how large the categorical spaces are and whether the numeric variables show any direct relationship with `claim_status`. The signal is modest, but it is enough to show that the problem is structured rather than random.

In [ ]:
# Categorical diagnostics and simple correlations
category_profile = pd.DataFrame(
    {
        'feature': categorical_features,
        'unique_levels': [df_train[col].nunique() for col in categorical_features],
        'top_level_share': [df_train[col].value_counts(normalize=True).iloc[0] for col in categorical_features]
    }
).sort_values(['unique_levels', 'top_level_share'], ascending=[False, False])

corr_with_target = (
    df_train[numerical_features + [target_col]]
    .corr(numeric_only=True)[target_col]
    .drop(target_col)
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .rename('pearson_corr_with_claim_status')
    .to_frame()
)

display(category_profile)
display(corr_with_target.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    df_train[numerical_features + [target_col]].corr(numeric_only=True),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=ax
)
ax.set_title('Numeric correlation heatmap')
plt.tight_layout()


### 2d. EDA takeaway

At this point the direction is fairly clear. The data is heavily imbalanced, some numeric variables are strongly right-skewed, and several categorical fields have too many levels to handle casually. That is why the final modeling path uses WoE for stable encoding, IV for feature filtering, and SMOTE to improve recall on the minority class.

## 3. Feature engineering with WoE

After the initial EDA, the next issue is representation. The raw feature set mixes skewed numeric variables with categorical fields that have very different cardinalities. WoE helps by translating those variables into a format that is easier to use in a linear decision model and easier to explain afterward.

In [ ]:
# 3. Feature Engineering: WoE transformation
fitted_binning_num = {}
fitted_binning_cat = {}

X_woe_num = pd.DataFrame(index=X_train.index)
X_woe_cat = pd.DataFrame(index=X_train.index)

for feature in numerical_features:
    optb = OptimalBinning(name=feature, dtype='numerical', solver='cp')
    optb.fit(X_train[feature].fillna(-999).values, y_train.values)
    fitted_binning_num[feature] = optb
    X_woe_num[f"{feature}_woe"] = optb.transform(X_train[feature].fillna(-999).values, metric='woe')

for feature in categorical_features:
    optb = OptimalBinning(name=feature, dtype='categorical', solver='cp')
    optb.fit(X_train[feature].astype(str).fillna('missing').values, y_train.values)
    fitted_binning_cat[feature] = optb
    X_woe_cat[f"{feature}_woe"] = optb.transform(X_train[feature].astype(str).fillna('missing').values, metric='woe')

X_train_woe = pd.concat([X_woe_num, X_woe_cat], axis=1)


### 3a. What the WoE stage produced

At this stage I want to check whether the transformation created a manageable feature space and whether some variables immediately stand out as more informative than others. Two quick views help here: the number of bins assigned to each feature, and the IV attached to the WoE binning itself.

In [ ]:
# WoE transformation overview
woe_feature_summary = []
all_binnings = {**fitted_binning_num, **fitted_binning_cat}

for feature, optb in all_binnings.items():
    binning_table = optb.binning_table.build().copy()
    bin_labels = binning_table['Bin'].astype(str)
    regular_bins = binning_table.loc[~bin_labels.isin(['Special', 'Missing', 'Totals'])]

    woe_feature_summary.append(
        {
            'feature': feature,
            'feature_type': 'numerical' if feature in numerical_features else 'categorical',
            'n_regular_bins': len(regular_bins),
            'iv': float(binning_table['IV'].iloc[-1])
        }
    )

woe_feature_summary = pd.DataFrame(woe_feature_summary).sort_values(['iv', 'n_regular_bins'], ascending=[False, False])
display(woe_feature_summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_iv = woe_feature_summary.head(10).sort_values('iv')
sns.barplot(data=top_iv, x='iv', y='feature', hue='feature_type', dodge=False, ax=axes[0])
axes[0].set_title('Top IV values after WoE binning')
axes[0].set_xlabel('Information Value')
axes[0].set_ylabel('feature')
axes[0].legend(title='feature type')

sns.scatterplot(data=woe_feature_summary, x='n_regular_bins', y='iv', hue='feature_type', s=90, ax=axes[1])
axes[1].set_title('Signal strength vs. bin complexity')
axes[1].set_xlabel('number of regular bins')
axes[1].set_ylabel('Information Value')

plt.tight_layout()

## 4. Feature selection with IV

Once the WoE variables are in place, I use Information Value to separate stronger predictors from weaker ones. This step is useful because it keeps the model leaner without throwing away the structure built during WoE transformation.

In [ ]:
# 4. Feature selection based on IV
features_to_drop = ['support_interactions', 'channel', 'person_age']
iv_series = pd.Series({feat: b.binning_table.build()['IV'].iloc[-1] for feat, b in {**fitted_binning_num, **fitted_binning_cat}.items()})
selected_features = iv_series[iv_series >= 0.02].index.tolist()
selected_features_filtered = [f for f in selected_features if f not in features_to_drop]
selected_features_woe = [f"{feat}_woe" for feat in selected_features_filtered]

X_train_woe = X_train_woe[selected_features_woe]


### 4a. Which features survived IV filtering

The IV threshold is only part of the decision. A few variables are still removed afterward because they add limited value or raise stability and interpretability concerns. That distinction is worth showing directly instead of only leaving it in code.

In [ ]:
# IV filtering summary
feature_selection_summary = woe_feature_summary.copy()
feature_selection_summary['passed_iv_threshold'] = feature_selection_summary['feature'].isin(selected_features)
feature_selection_summary['manually_removed'] = feature_selection_summary['feature'].isin(features_to_drop)
feature_selection_summary['kept_in_final_model'] = feature_selection_summary['feature'].isin(selected_features_filtered)

feature_selection_summary['status'] = np.select(
    [
        feature_selection_summary['kept_in_final_model'],
        feature_selection_summary['manually_removed'],
        feature_selection_summary['passed_iv_threshold']
    ],
    [
        'kept in final model',
        'removed after review',
        'passed threshold but excluded'
    ],
    default='below IV threshold'
)

display(
    feature_selection_summary[
        ['feature', 'feature_type', 'n_regular_bins', 'iv', 'status']
    ].sort_values(['iv', 'feature'], ascending=[False, True]).round(3)
)

fig, ax = plt.subplots(figsize=(10, 6))
plot_df = feature_selection_summary.sort_values('iv', ascending=False).head(12).sort_values('iv')
sns.barplot(data=plot_df, x='iv', y='feature', hue='status', dodge=False, ax=ax)
ax.set_title('How IV filtering narrowed the feature set')
ax.set_xlabel('Information Value')
ax.set_ylabel('feature')
ax.legend(title='selection status', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

### 4b. Did IV filtering actually help?

This is the more important question. A feature-selection step is only useful if it improves the trade-off or simplifies the model without harming performance. The chart below uses the report-backed validation results from the project to compare all WoE features against the low-IV-pruned version.

In [ ]:
# Report-backed feature selection comparison
iv_report = pd.DataFrame([
    {'model': 'ElasticNet', 'feature_set': 'All WoE features', 'AUROC': 0.828, 'Recall': 0.697, 'Balanced Accuracy': 0.759},
    {'model': 'ElasticNet', 'feature_set': 'Low-IV removed', 'AUROC': 0.830, 'Recall': 0.709, 'Balanced Accuracy': 0.758},
    {'model': 'Logistic Regression', 'feature_set': 'All WoE features', 'AUROC': 0.828, 'Recall': 0.697, 'Balanced Accuracy': 0.759},
    {'model': 'Logistic Regression', 'feature_set': 'Low-IV removed', 'AUROC': 0.830, 'Recall': 0.709, 'Balanced Accuracy': 0.758}
])

display(iv_report.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
metrics_to_plot = ['AUROC', 'Recall', 'Balanced Accuracy']

for ax, metric in zip(axes, metrics_to_plot):
    sns.barplot(data=iv_report, x='model', y=metric, hue='feature_set', ax=ax)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=10)
    if metric != 'AUROC':
        ax.set_ylim(0.65, 0.85)

axes[0].legend(title='feature set', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].legend_.remove()
axes[2].legend_.remove()
plt.tight_layout()

## 5. Modeling and why the final model was chosen

This section pulls together the two practical decisions that matter most after feature engineering: how to keep the feature set disciplined, and how to choose a model that handles imbalance without becoming hard to justify.

In [ ]:
# 5. Transform the test set with the same WoE bins
def transform_woe(df, binning_num, binning_cat):
    X_num = pd.DataFrame(index=df.index)
    X_cat = pd.DataFrame(index=df.index)
    for feature, optb in binning_num.items():
        X_num[f"{feature}_woe"] = optb.transform(df[feature].fillna(-999).values, metric='woe')
    for feature, optb in binning_cat.items():
        X_cat[f"{feature}_woe"] = optb.transform(df[feature].astype(str).fillna('missing').values, metric='woe')
    return pd.concat([X_num, X_cat], axis=1)

X_test_woe = transform_woe(df_test, fitted_binning_num, fitted_binning_cat)
X_test_woe = X_test_woe[selected_features_woe]


### 5a. Other models that were tried before finalizing Elastic Net

The final model did not appear in isolation. Earlier runs compared Elastic Net with standard Logistic Regression and SVM with an RBF kernel. The goal was not just to find the highest score on one metric, but to find a model that stayed competitive on validation data and remained interpretable enough for an insurance decision setting.

In [ ]:
# Report-backed candidate model comparison
candidate_models = pd.DataFrame([
    {'model': 'ElasticNet', 'dataset': 'Training', 'Accuracy': 0.798, 'Balanced Accuracy': 0.746, 'Recall': 0.692, 'F1': 0.092, 'ROC_AUC': 0.812},
    {'model': 'ElasticNet', 'dataset': 'Validation', 'Accuracy': 0.797, 'Balanced Accuracy': 0.732, 'Recall': 0.664, 'F1': 0.088, 'ROC_AUC': 0.782},
    {'model': 'Logistic Regression (class weight)', 'dataset': 'Training', 'Accuracy': 0.768, 'Balanced Accuracy': 0.731, 'Recall': 0.694, 'F1': 0.086, 'ROC_AUC': 0.796},
    {'model': 'Logistic Regression (class weight)', 'dataset': 'Validation', 'Accuracy': 0.770, 'Balanced Accuracy': 0.714, 'Recall': 0.656, 'F1': 0.083, 'ROC_AUC': 0.773},
    {'model': 'Logistic Regression (SMOTE)', 'dataset': 'Training', 'Accuracy': 0.798, 'Balanced Accuracy': 0.746, 'Recall': 0.692, 'F1': 0.092, 'ROC_AUC': 0.812},
    {'model': 'Logistic Regression (SMOTE)', 'dataset': 'Validation', 'Accuracy': 0.798, 'Balanced Accuracy': 0.732, 'Recall': 0.664, 'F1': 0.088, 'ROC_AUC': 0.782},
    {'model': 'SVM (RBF)', 'dataset': 'Training', 'Accuracy': 0.769, 'Balanced Accuracy': 0.783, 'Recall': 0.798, 'F1': 0.092, 'ROC_AUC': 0.850},
    {'model': 'SVM (RBF)', 'dataset': 'Validation', 'Accuracy': 0.766, 'Balanced Accuracy': 0.712, 'Recall': 0.656, 'F1': 0.076, 'ROC_AUC': 0.738}
])

display(candidate_models.round(3))

validation_view = candidate_models[candidate_models['dataset'] == 'Validation'].copy()
roc_gap = candidate_models.pivot(index='model', columns='dataset', values='ROC_AUC').reset_index()
roc_gap['train_validation_gap'] = roc_gap['Training'] - roc_gap['Validation']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=validation_view, x='model', y='ROC_AUC', ax=axes[0])
axes[0].set_title('Validation ROC-AUC across candidate models')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(data=roc_gap.sort_values('train_validation_gap', ascending=False), x='model', y='train_validation_gap', ax=axes[1])
axes[1].set_title('Train vs. validation ROC-AUC gap')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylabel('ROC-AUC gap')

plt.tight_layout()

### 5b. Why rebalancing mattered more than headline accuracy

One hidden trap in this dataset is that a model can look accurate while still doing almost nothing for the positive class. That is why I compare rebalancing methods directly using recall and balanced accuracy instead of only looking at accuracy.

In [ ]:
# Report-backed rebalancing comparison
rebalancing_report = pd.DataFrame([
    {'model': 'Logistic Regression', 'rebalancing': 'Random under-sampling', 'AUROC': 0.828, 'Accuracy': 0.819, 'Precision': 0.055, 'Recall': 0.697, 'F1': 0.101, 'Balanced Accuracy': 0.759},
    {'model': 'Logistic Regression', 'rebalancing': 'SMOTE', 'AUROC': 0.830, 'Accuracy': 0.823, 'Precision': 0.056, 'Recall': 0.701, 'F1': 0.103, 'Balanced Accuracy': 0.762},
    {'model': 'ElasticNet', 'rebalancing': 'Random under-sampling', 'AUROC': 0.828, 'Accuracy': 0.819, 'Precision': 0.055, 'Recall': 0.697, 'F1': 0.101, 'Balanced Accuracy': 0.759},
    {'model': 'ElasticNet', 'rebalancing': 'SMOTE', 'AUROC': 0.830, 'Accuracy': 0.823, 'Precision': 0.056, 'Recall': 0.701, 'F1': 0.103, 'Balanced Accuracy': 0.762}
])

display(rebalancing_report.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=rebalancing_report, x='model', y='Recall', hue='rebalancing', ax=axes[0])
axes[0].set_title('Recall by model and rebalancing method')
axes[0].set_xlabel('')

sns.barplot(data=rebalancing_report, x='model', y='Balanced Accuracy', hue='rebalancing', ax=axes[1])
axes[1].set_title('Balanced accuracy by model and rebalancing method')
axes[1].set_xlabel('')

axes[0].legend(title='rebalancing')
axes[1].legend(title='rebalancing')
plt.tight_layout()

### 5c. Why the final model was Elastic Net

A few reasons made the final choice more convincing than simply picking the top score on one chart:

- **Accuracy was not the deciding metric.** In a dataset with only about 1.46% positives, it is easy to post a high accuracy number while missing most approved claims.
- **SVM was explored but did not generalize as cleanly.** Its training ROC-AUC was the strongest, but its validation drop was noticeably larger, which is exactly the kind of pattern I wanted to avoid in a final portfolio model.
- **Elastic Net stayed competitive while being easier to justify.** Its validation performance was essentially tied with Logistic Regression plus SMOTE, but the regularization made it a safer choice once the feature set included multiple related WoE-transformed predictors.
- **Low-IV pruning helped without changing the story.** The feature-selection results suggest the simpler version of the model preserved performance and slightly improved recall and AUROC.

That combination of stability, interpretability, and competitive validation performance is what made Elastic Net the final recommendation.

## 6. Load the tuned Elastic Net setup

After comparing candidate models and rebalancing strategies, I keep Elastic Net with SMOTE as the final setup. The saved parameters below reflect that final choice without re-running the full tuning workflow in this notebook.

In [ ]:
# 6. Load tuned Elastic Net settings
with open(config_dir / "parameter" / "ElasticNet_best_params.json") as f:
    best_params = json.load(f)

with open(config_dir / "parameter" / "ElasticNet_youden_threshold.json") as f:
    youden_thresh = json.load(f)['youden_threshold']

# Build the model pipeline
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(solver='saga', penalty='elasticnet', max_iter=1000, random_state=42))
])

# Set the tuned hyperparameters
pipeline.set_params(**best_params)


In [ ]:
# 7. Train the final model and export predictions
pipeline.fit(X_train_woe, y_train)

# Save the trained model artifact
joblib.dump(pipeline, config_dir / 'classification_model.pkl')

# Predict on the holdout test set
y_test_scores = pipeline.predict_proba(X_test_woe)[:, 1]
y_test_pred = (y_test_scores >= youden_thresh).astype(int)

if 'observation_id' not in df_test.columns:
    df_test['observation_id'] = df_test.index + 1

df_predictions = df_test[['observation_id']].copy()
df_predictions['predicted_claim_status'] = y_test_pred
df_predictions.to_csv(output_dir / 'insurance_predictions.csv', index=False)

print("Predictions saved to 'outputs/insurance_predictions.csv'.")


## Final takeaway

The strongest part of this project is not just that Elastic Net produced solid metrics. The more convincing part is that each modeling decision follows from what the data looked like in the first place: a rare positive class, skewed numeric variables, and high-cardinality categorical fields.

That is also why the final model is not presented as the only model that worked. Logistic Regression remained competitive, SVM was explored and set aside, and low-IV pruning was tested rather than assumed. The final pipeline earned its place through comparison, not by default.